<a href="https://colab.research.google.com/github/MaikaelLubis/Capstone_Training_Via_Google_Colabs/blob/main/Logbook_Week8th.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Sun Nov 22 14:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install Ultralytics & Tarik Dataset dari Robofflow

In [3]:
!pip install -q ultralytics roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="rWjZbG7mkvUeUA15w11f")
projects = project = rf.workspace("maikaels-workspace").project("cctv_ws_sanindo")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CCTV_WS_SANINDO-1 in yolov11:: 100%|██████████| 3377/3377 [00:00<00:00, 6814.28it/s]


Mulai Training YOLOv11s

In [6]:
from ultralytics import YOLO

# Model
model = YOLO("yolo11s.pt")

# Mulai training process
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=1024,
    batch=16,
    patience=25,
    project="/content/drive/MyDrive/OCCLU_TRACK_Models",
    name="train_yolov11s_1024",
    exist_ok=True
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/CCTV_WS_SANINDO-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_yolov11s_1024, nbs=64, nms=None, opset=None, 

EVALUASI pada Test Set (October 2nd)

In [7]:
import glob
import os
from google.colab import drive
from ultralytics import YOLO

In [8]:
# Google Drive harus tetap terhubung

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

In [12]:
# Cari otomatis file data.yaml dari folder dataset Roboflow

yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
if not yaml_files:
  raise FileNotFoundError(
      'Folder dataset belum ada di sesi ini. Jalankan ulang Cell 2 (snipper'
      'roboflow) dulu, lalu jalankan cell ini lagi. '
  )
data_yaml_path = yaml_files[0]


In [14]:
# Muat bobot terbaik (best.pt) hasil training 100 Epoch

best_model_path = '/content/drive/MyDrive/OCCLU_TRACK_Models/train_yolov11s_1024/weights/best.pt'
model = YOLO(best_model_path)

In [15]:
# Jalankan ujian akhir pada Test Set (October 2nd: Siang & Malam)

test_results = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=1024,
    batch=16,
    project='/content/drive/MyDrive/OCCLU_TRACK_Models',
    name='test_eval_yolo11s_1024',
    exist_ok=True,
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 16.4±6.7 MB/s, size: 44.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/CCTV_WS_SANINDO-1/test/labels... 292 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 469.4it/s 0.6s
val: New cache created: /content/CCTV_WS_SANINDO-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.2s
                   all        292       1896       0.87      0.835      0.874      0.462
Speed: 2.3ms preprocess, 13.6ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/drive/MyDrive/OCCLU_TRACK_Models/test_eval_yolo11s_1024


##**INI CELL YANG SAMA UNTUK MENYIMPAN HASIL DI CAPSTONE**

In [17]:
import glob
import os
import shutil
from google.colab import drive
from ultralytics import YOLO

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
if not yaml_files:
  raise FileNotFoundError(
      'Folder dataset belum ada. Jalankan ulang sel snippet Roboflow dulu.'
  )

data_yaml_path = yaml_files[0]
best_model_path = '/content/drive/MyDrive/OCCLU_TRACK_Models/train_yolov11s_1024/weights/best.pt'
model = YOLO(best_model_path)

# 1. Jalankan evaluasi Test Set dan simpan langsung ke folder PAPER_CAPSTONE
test_results = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=1024,
    batch=16,
    project='/content/drive/MyDrive/PAPER_CAPSTONE/Gambar Results DI Screenshoot',
    name='Hasil_Test_2Oktober',
    exist_ok=True,
)

# 2. Salin grafik kurva training (100 epoch) ke PAPER_CAPSTONE untuk bahan bab 4
train_results_img = '/content/drive/MyDrive/OCCLU_TRACK_Models/train_yolo11s_1024/results.png'
if os.path.exists(train_results_img):
  shutil.copy(
      train_results_img,
      '/content/drive/MyDrive/PAPER_CAPSTONE/Kurva_Training_100Epoch.png',
  )

print('Selesai! Semua gambar sudah tersimpan di Google Drive > PAPER_CAPSTONE')

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1687.1±597.5 MB/s, size: 44.7 KB)
val: Scanning /content/CCTV_WS_SANINDO-1/test/labels.cache... 292 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 58.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.4s
                   all        292       1896       0.87      0.835      0.874      0.462
Speed: 2.6ms preprocess, 13.6ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/drive/MyDrive/PAPER_CAPSTONE/Gambar Results DI Screenshoot/Hasil_Test_2Oktober
Selesai! Semua gambar sudah tersimpan di Google Drive > PAPER_CAPSTONE
